# Kalshi API – PoC Notebook

In [1]:
import os
import json
import base64
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timezone
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))

KALSHI_KEY_ID   = os.getenv('KALSHI_KEY_ID')
KALSHI_KEY_PATH = os.getenv('KALSHI_PRIVATE_KEY_PATH')
BASE_URL        = 'https://api.elections.kalshi.com/trade-api/v2'

assert KALSHI_KEY_ID,   'KALSHI_KEY_ID not found in .env'
assert KALSHI_KEY_PATH, 'KALSHI_PRIVATE_KEY_PATH not found in .env'

_pem_path = os.path.join(os.path.dirname(os.getcwd()), KALSHI_KEY_PATH)
with open(_pem_path, 'rb') as f:
    _private_key = serialization.load_pem_private_key(f.read(), password=None)

def kalshi_headers(method: str, path: str) -> dict:
    ts  = str(int(datetime.now(timezone.utc).timestamp() * 1000))
    msg = (ts + method.upper() + path).encode()
    sig = _private_key.sign(
        msg,
        asym_padding.PSS(
            mgf=asym_padding.MGF1(hashes.SHA256()),
            salt_length=asym_padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    return {
        'KALSHI-ACCESS-KEY':       KALSHI_KEY_ID,
        'KALSHI-ACCESS-TIMESTAMP': ts,
        'KALSHI-ACCESS-SIGNATURE': base64.b64encode(sig).decode(),
    }

print('Auth ready.')

Auth ready.


## Step 1 — Browse Series (Sport Categories)
Run this to find the `series_ticker` for the sport you want (e.g. soccer, NBA, NFL).

In [2]:
PATH = '/trade-api/v2/series'
category = "Sports"
resp = requests.get(f'{BASE_URL}/series', headers=kalshi_headers('GET', PATH))
resp.raise_for_status()

def mask(event, category: str = 'Sports'):
    return event.get('category') == category

series = resp.json().get('series', [])
if category: 
    df_series = pd.DataFrame(filter(mask, [{'ticker': s['ticker'], 'title': s.get('title'), 'category': s.get('category')} for s in series]))
else:
    df_series = pd.DataFrame([{'ticker': s['ticker'], 'title': s.get('title'), 'category': s.get('category')} for s in series])

# Show all categories so you can see what's available
print(df_series['category'].value_counts().to_string())
print()
df_series

category
Sports    1619



,ticker,title,category
0,KXNBADRAFT7,NBA Draft Seventh Pick,Sports
1,KXNFLCELEBRITYGAME,Pro Football Celebrity Flag Football Game,Sports
2,KXUFCMIDDLEWEIGHTTITLE,UFC Middleweight Title,Sports
3,KXUCL16RMAMCI,Champions League round of 16 - RMA MCI,Sports
4,KXSLGREECEGAME,Super League Greece Game,Sports
...,...,...,...
1614,KXCOACHOUTNCAAFB,NCAAFB Coaches Out,Sports
1615,KXNEWCOACHUNC,University of North Carolina new coach,Sports
1616,KXATPCHALLENGERMATCH,Challenger ATP,Sports
1617,KXWTAGRANDSLAM,WTA Grand Slam,Sports


## Step 2 — Load All Contracts for a Series, Filter by Team

In [3]:
SERIES_TICKER = 'KXARGPREMDIVGAME'          # from Step 1
TEAMS         = None  # ['hawks', 'detroit']  # all terms must match, leave [] for all contracts

PATH = '/trade-api/v2/markets'
resp = requests.get(
    f'{BASE_URL}/markets',
    headers=kalshi_headers('GET', PATH),
    params={'series_ticker': SERIES_TICKER, 'status': 'open', 'limit': 100},
)
resp.raise_for_status()

markets = resp.json().get('markets', [])
if TEAMS:
    for team in TEAMS:
        markets = [m for m in markets if team.lower() in m.get('title', '').lower()
                                    or team.lower() in m.get('yes_sub_title', '').lower()]

print(f'{len(markets)} contracts matched')

pd.DataFrame([{
    'event_ticker': m.get('event_ticker'),
    'ticker':       m['ticker'],
    'title':        m.get('title'),
    'yes_sub_title': m.get('yes_sub_title'),
    'yes_bid':      m.get('yes_bid'),
    'yes_ask':      m.get('yes_ask'),
    'no_bid':       m.get('no_bid'),
    'no_ask':       m.get('no_ask'),
    'volume':       m.get('volume'),
    'open_int':     m.get('open_interest'),
} for m in markets])

3 contracts matched


,event_ticker,ticker,title,yes_sub_title,yes_bid,yes_ask,no_bid,no_ask,volume,open_int
0,KXARGPREMDIVGAME-26MAR26ARGLAN,KXARGPREMDIVGAME-26MAR26ARGLAN-TIE,Argentinos Juniors vs Lanus Winner?,Tie,None,None,None,None,None,None
1,KXARGPREMDIVGAME-26MAR26ARGLAN,KXARGPREMDIVGAME-26MAR26ARGLAN-LAN,Argentinos Juniors vs Lanus Winner?,Lanus,None,None,None,None,None,None
2,KXARGPREMDIVGAME-26MAR26ARGLAN,KXARGPREMDIVGAME-26MAR26ARGLAN-ARG,Argentinos Juniors vs Lanus Winner?,Argentinos Juniors,None,None,None,None,None,None


## Step 3 — Contracts Within an Event
Set `EVENT_TICKER` to a row from Step 2. Each row is one binary contract (e.g. "Barcelona to win").

In [4]:
EVENT_TICKER = 'KXNBASOUTHEAST-25'

PATH = f'/trade-api/v2/events/{EVENT_TICKER}'
resp = requests.get(f'{BASE_URL}/events/{EVENT_TICKER}', headers=kalshi_headers('GET', PATH))
resp.raise_for_status()

raw       = resp.json()
event     = raw.get('event', {})
contracts = raw.get('markets', [])

print(f"Event : {event.get('title')}")
print(f"Contracts: {len(contracts)}")

pd.DataFrame([{
    'ticker':   c['ticker'],
    'title':    c.get('yes_sub_title'),
    'yes_bid':  c.get('yes_bid'),
    'yes_ask':  c.get('yes_ask'),
    'yes_mid':  round((c.get('yes_bid', 0) + c.get('yes_ask', 0)) / 2, 1),
    'no_bid':   c.get('no_bid'),
    'no_ask':   c.get('no_ask'),
    'no_mid':   round((c.get('no_bid', 0) + c.get('no_ask', 0)) / 2, 1),
    'volume':   c.get('volume'),
    'open_int': c.get('open_interest'),
} for c in contracts])

Event : Pro Basketball Southeast Division Winner
Contracts: 5


,ticker,title,yes_bid,yes_ask,yes_mid,no_bid,no_ask,no_mid,volume,open_int
0,KXNBASOUTHEAST-25-ORL,Orlando,None,None,0.0,None,None,0.0,None,None
1,KXNBASOUTHEAST-25-MIA,Miami,None,None,0.0,None,None,0.0,None,None
2,KXNBASOUTHEAST-25-ATL,Atlanta,None,None,0.0,None,None,0.0,None,None
3,KXNBASOUTHEAST-25-CHA,Charlotte,None,None,0.0,None,None,0.0,None,None
4,KXNBASOUTHEAST-25-WAS,Washington,None,None,0.0,None,None,0.0,None,None
